# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/suha-2004/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 234 (delta 116), reused 102 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (234/234), 3.07 MiB | 9.55 MiB/s, done.
Resolving deltas: 100% (116/116), done.


In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
!ls data/raw

content_refresh_anonymized.csv


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*
### Distribution observation

The key search-performance and content fields were inspected using summary statistics and mean-versus-median comparisons. Count-based fields such as impressions, clicks, and sessions may show skewed distributions because a small number of content items can receive substantially more activity than the typical item. These distributions are therefore considered when interpreting the signal tests.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Key numerical fields for the signal audit

audit_cols = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "engagement_rate"
]

# Summary statistics
distribution_summary = df[audit_cols].describe().T

distribution_summary

,count,mean,std,min,25%,50%,75%,max
impressions_prev_30d,30000.0,1783.078500,6150.429511,0.0,19.0,210.00,1143.00,218786.0
clicks_prev_30d,30000.0,5.435100,28.358673,0.0,0.0,0.00,2.00,1627.0
sessions_prev_30d,30000.0,10.283000,42.578003,0.0,1.0,2.00,7.00,4247.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
engagement_rate,30000.0,2.534520,8.310096,0.0,0.0,0.00,1.35,100.0


In [6]:
# Compare mean and median

heavy_tail_check = pd.DataFrame({
    "Feature": audit_cols,
    "Mean": [df[col].mean() for col in audit_cols],
    "Median": [df[col].median() for col in audit_cols]
})

heavy_tail_check["Mean_to_Median_Ratio"] = (
    heavy_tail_check["Mean"] /
    heavy_tail_check["Median"].replace(0, np.nan)
)

heavy_tail_check

,Feature,Mean,Median,Mean_to_Median_Ratio
0,impressions_prev_30d,1783.078500,210.00,8.490850
1,clicks_prev_30d,5.435100,0.00,NaN
2,sessions_prev_30d,10.283000,2.00,5.141500
3,content_age_days,256.167800,236.00,1.085457
4,days_since_last_update,46.098300,20.00,2.304915
5,ctr,0.510733,0.07,7.296190
6,engagement_rate,2.534520,0.00,NaN


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*
### Signal test #1 — Previous impressions

**Verdict: CONFIRMED**

Recent-period impressions increase strongly across groups with higher previous-period impressions. The mean recent impressions increase from 52.80 in the lowest previous-impression group to 5159.75 in the highest group. This supports the directional assumption that historical impressions are informative about later search visibility.
### Signal test #2 — Previous clicks

**Verdict: CONFIRMED**

Recent-period clicks are substantially higher for content with higher previous-period clicks. The mean recent clicks increase from 0.49 in the lower previous-click group to 20.36 in the higher group. This supports the directional assumption that historical click activity is informative about later click activity.
### Signal test #3 — Content freshness

**Verdict: MIXED**

Recent-period impressions do not show a simple monotonic relationship with days since the last update. The mean increases from 1068.79 for content updated within 20 days to 1867.74 for content updated 20–104 days ago, but falls to 345.08 for content not updated for more than 104 days. This suggests that freshness may be informative, but the relationship is not simple and should be treated as directional evidence rather than a fixed rule.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal test #1:
# Previous-period impressions vs recent-period impressions

signal_1 = df[
    ["impressions_prev_30d", "impressions_last_30d"]
].copy()

signal_1["prev_impressions_group"] = pd.qcut(
    signal_1["impressions_prev_30d"],
    q=4,
    duplicates="drop"
)

signal_1_summary = (
    signal_1
    .groupby("prev_impressions_group", observed=True)
    ["impressions_last_30d"]
    .agg(["count", "mean", "median"])
)

signal_1_summary

,count,mean,median
prev_impressions_group,,,
"(-0.001, 19.0]",7521,52.803085,1.0
"(19.0, 210.0]",7489,76.419015,48.0
"(210.0, 1143.0]",7491,428.432386,310.0
"(1143.0, 218786.0]",7499,5159.746766,2249.0


In [8]:
# Signal test #2:
# Previous-period clicks vs recent-period clicks

signal_2 = df[
    ["clicks_prev_30d", "clicks_last_30d"]
].copy()

signal_2["prev_clicks_group"] = pd.qcut(
    signal_2["clicks_prev_30d"],
    q=4,
    duplicates="drop"
)

signal_2_summary = (
    signal_2
    .groupby("prev_clicks_group", observed=True)
    ["clicks_last_30d"]
    .agg(["count", "mean", "median"])
)

signal_2_summary

,count,mean,median
prev_clicks_group,,,
"(-0.001, 2.0]",23295,0.492509,0.0
"(2.0, 1627.0]",6705,20.364355,8.0


In [9]:
# Signal test #3:
# Days since last update vs recent impressions

signal_3 = df[
    ["days_since_last_update", "impressions_last_30d"]
].copy()

signal_3["update_age_group"] = pd.qcut(
    signal_3["days_since_last_update"],
    q=4,
    duplicates="drop"
)

signal_3_summary = (
    signal_3
    .groupby("update_age_group", observed=True)
    ["impressions_last_30d"]
    .agg(["count", "mean", "median"])
)

signal_3_summary

,count,mean,median
update_age_group,,,
"(0.999, 20.0]",15866,1068.785768,56.0
"(20.0, 104.0]",13816,1867.738347,267.0
"(104.0, 373.0]",318,345.075472,7.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*
### Flag-linked test — Trend direction

**Verdict: CONFIRMED**

Recent-period impressions differ substantially across the observed trend-direction categories. Content marked as `stable` has the highest mean recent impressions at 2962.58, followed by `up` at 2173.77, while `down`, `new`, and `flat` have lower mean values. This supports the directional assumption that the trend-direction flag captures measurable differences in recent search visibility, although it does not establish causation.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag-linked test:
# Check whether trend_direction is associated with recent impressions

flag_test = (
    df.groupby("trend_direction", dropna=False)
      ["impressions_last_30d"]
      .agg(["count", "mean", "median"])
      .sort_values("mean", ascending=False)
)

flag_test

,count,mean,median
trend_direction,,,
stable,5962,2962.575310,543.0
up,4388,2173.773473,229.0
down,16262,941.556635,128.0
new,2236,160.454383,2.0
flat,1152,0.000000,0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*
### Practical interpretation

The signal audit shows that previous-period impressions and clicks are useful directional signals for understanding later search activity, while the freshness signal has a more mixed relationship with recent visibility. The trend-direction flag also shows measurable differences in recent impressions across its categories. A content team can use these signals as decision-support evidence for prioritizing content for review, while avoiding the assumption that any single signal guarantees future performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.